In [1]:
import os

import albumentations
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

In [2]:
model_path = "../input/rn-wc-tta-calr/model(6).pth"
sample_sub_path = "../input/cassava-leaf-disease-classification/sample_submission.csv"
test_images_path = "../input/cassava-leaf-disease-classification/test_images"

# Creating model and loading weights

In [3]:
# Loading ResNet from PyTorch and update it's fc layer output to 5 classes
model = models.resnext50_32x4d(pretrained=False)
model.fc = nn.Linear(2048, 5)

model.to(torch.device("cuda:0" if torch.cuda.is_available() else "cpu"))
model.load_state_dict(torch.load(model_path))
model.eval()

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


FileNotFoundError: [Errno 2] No such file or directory: '../input/rn-wc-tta-calr/model(6).pth'

# Submission Augmentations

In [4]:
sub_aug = albumentations.Compose([
            albumentations.RandomResizedCrop(256, 256),
            albumentations.Transpose(p=0.5),
            albumentations.HorizontalFlip(p=0.5),
            albumentations.VerticalFlip(p=0.5),
            albumentations.ShiftScaleRotate(p=0.5),
            albumentations.HueSaturationValue(
                hue_shift_limit=0.2, 
                sat_shift_limit=0.2, 
                val_shift_limit=0.2, 
                p=0.5
            ),
            albumentations.RandomBrightnessContrast(
                brightness_limit=(-0.1,0.1), 
                contrast_limit=(-0.1, 0.1), 
                p=0.5
            ),
            albumentations.Normalize(
                mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225], 
                max_pixel_value=255.0, 
                p=1.0
            ),
            albumentations.CoarseDropout(p=0.5),
            albumentations.Cutout(p=0.5)], p=1.)

ValueError: 2 validation errors for InitSchema
scale
  Input should be a valid tuple [type=tuple_type, input_value=256, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/tuple_type
size
  Input should be a valid tuple [type=tuple_type, input_value=256, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/tuple_type

# Creating the Submission file

In [5]:
sample_sub = pd.read_csv(sample_sub_path)

predictions = []
for _, sample_row in sample_sub.iterrows():
    
    image_pred = 0
    for j in range(10):
        image = np.array(Image.open(os.path.join(test_images_path, sample_row.image_id)))
        image = sub_aug(image=image)["image"]
        image = transforms.ToTensor()(np.array(image))
        image = image.to(torch.device("cuda:0" if torch.cuda.is_available() else "cpu"))
        outputs = model(image.unsqueeze(0))
        
        image_pred += outputs
    image_pred /= 10
    _, pred_label = torch.max(image_pred, 1)
        
    predictions.append([sample_row.image_id, pred_label.item()])

sub_df = pd.DataFrame(predictions,columns=['image_id', 'label'])
sub_df.to_csv('submission.csv', index=False)
print(sub_df.head())


NameError: name 'sub_aug' is not defined